# VCARE CC Compass: live-app test (Option 2)

**What this does, in plain terms:** the same 10-question test as Option 1, but this time the AI assistant reaches the tools through the **real deployed app** (`vcare-pathway-checklist`) over the internet, signed in as the hackathon test account **Inspector Gadget**.

**Analogy:** Option 1 was a practice exam in the classroom. This is the real exam at the testing center.

**How to run:** choose **Run all**. It takes about 3 to 5 minutes.

**Requirements (already done):** the app is shared with Inspector Gadget with **Can use**. If you see HTTP 401, check that permission under Compute, Apps, `vcare-pathway-checklist`, Permissions.

*All data is synthetic, including consult numbers. Results are test priorities, not clinical determinations.*

In [0]:
# Installs the MCP client and scorecard libraries, then restarts Python so they load cleanly.
%pip install -q --upgrade "databricks-mcp" "databricks-sdk>=0.60" "mlflow>=3.4" "openai>=1.40" nest_asyncio
%restart_python

In [0]:
# Settings for this run. Change these values, not the code below.
APP_URL = "https://vcare-pathway-checklist-7474654188777746.aws.databricksapps.com"  # the app's URL, WITHOUT /mcp
REPO = "/Workspace/Users/anna.fernandez@va.gov/mcp-vcare-compass"
GOLD_SET = f"{REPO}/eval/phoenix/datasets/mcp_vcare_eval_0/evaluation.xml"  # same 10 questions as Option 1
EXPERIMENT = "/Users/anna.fernandez@va.gov/mcp_vcare_compass_eval"      # same experiment as Option 1, so runs sit side by side
AGENT_MODEL = "databricks-claude-sonnet-4-5"                          # same assistant as Option 1
RUN_NAME = "mcp_vcare_eval_0 option2-live"

EXPECTED_TOOL_NAMES = ["ask_vcare_case_data", "flag_high_risk_case", "get_consult_pathway_checklist"]
EXPECTED_TOOL = {
    1: "ask_vcare_case_data", 2: "ask_vcare_case_data", 3: "ask_vcare_case_data",
    4: "ask_vcare_case_data", 5: "ask_vcare_case_data",
    6: "get_consult_pathway_checklist", 7: "get_consult_pathway_checklist",
    8: "flag_high_risk_case", 9: "flag_high_risk_case", 10: "flag_high_risk_case",
}

# Same standing instructions as Option 1, so the only change is HOW the tools are reached.
SYSTEM_PROMPT = (
    "You answer questions about VA Community Care consult review using ONLY the provided tools. "
    "Call the tools you need, then reply with ONLY the final answer in the exact format the question "
    "asks for: no explanation, no working, no extra words."
)
import os
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"   # one question at a time over the network, for stability

In [0]:
# Signs in as the hackathon test account and lists the tools the LIVE app offers.
# The sign-in secret stays in Unity Catalog (models.ai / inspector-gadget); it is never printed.
import json, time
from urllib.parse import urljoin
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
import nest_asyncio
nest_asyncio.apply()

def inspector_gadget_client() -> WorkspaceClient:
    params = json.loads(dbutils.secrets.get(catalog="models", schema="ai", key="inspector-gadget"))
    host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
    return WorkspaceClient(host=host, client_id=params["client_id"], client_secret=params["client_secret"])

ig_client = inspector_gadget_client()
SERVER_URL = urljoin(APP_URL, "mcp")
mcp_client = DatabricksMCPClient(server_url=SERVER_URL, workspace_client=ig_client)
live_tools = mcp_client.list_tools()
names = sorted(t.name for t in live_tools)
print("Tools on the live app:", names)
assert names == sorted(EXPECTED_TOOL_NAMES), f"Expected {EXPECTED_TOOL_NAMES}, got {names}. Is the app running the GitHub version?"
print("PASS  all 3 tools are reachable on the live app")

# Tool calls use the standard MCP client with longer timeouts (120 s to connect/send, 300 s to read):
# Genie needs 10-40 s to write and run its query, longer than DatabricksMCPClient's ~5 s default.
import asyncio
from datetime import timedelta
import httpx
import mcp.client.streamable_http as _streamable_http
from mcp import ClientSession

def _auth_headers():
    """A fresh Inspector Gadget sign-in header (OAuth token) for each call; never printed."""
    return ig_client.config.authenticate()

async def _session_call(streams, name, args):
    async with ClientSession(streams[0], streams[1]) as session:
        await session.initialize()
        return await session.call_tool(name, args)

async def _call_tool_async(name, args):
    if hasattr(_streamable_http, "streamablehttp_client"):   # older mcp library
        async with _streamable_http.streamablehttp_client(SERVER_URL, headers=_auth_headers(), timeout=timedelta(seconds=120),
                                                          sse_read_timeout=timedelta(seconds=300)) as streams:
            return await _session_call(streams, name, args)
    # newer mcp library: the timeouts live on the web client
    async with httpx.AsyncClient(headers=_auth_headers(), timeout=httpx.Timeout(120.0, read=300.0), follow_redirects=True) as http:
        async with _streamable_http.streamable_http_client(SERVER_URL, http_client=http) as streams:
            return await _session_call(streams, name, args)

def call_tool_live(name, args):
    """Call one tool on the live app as Inspector Gadget, allowing up to 5 minutes for a slow reply."""
    return asyncio.run(_call_tool_async(name, args))

In [0]:
# One known-answer call per tool, straight to the live app (no AI assistant yet).
def call_live(name, args):
    """Call one tool on the live app. Returns (is_error, seconds, parsed_response)."""
    started = time.time()
    try:
        result = call_tool_live(name, args)
    except Exception as e:  # the MCP client wraps network errors in an ExceptionGroup; show the real cause
        inner = getattr(e, "exceptions", [e])
        return True, round(time.time() - started, 1), {"error": " | ".join(f"{type(x).__name__}: {x}" for x in inner)[:600]}
    seconds = round(time.time() - started, 1)
    text = result.content[0].text if result.content else ""
    try:
        data = json.loads(text)
    except ValueError:
        data = {"raw": text}
    return bool(getattr(result, "isError", getattr(result, "is_error", False))), seconds, data

smoke = []

err, s, out = call_live("get_consult_pathway_checklist", {"args": {"pathway_step": "schedule"}})
label = (out.get("steps") or [{}])[0].get("label")
smoke.append(("Checklist: 'schedule' step label is 'Schedule'", (not err) and label == "Schedule", s, label))

err, s, out = call_live("flag_high_risk_case", {"args": {
    "case_id": "SYN-EVAL-01", "category_of_care": "Oncology", "as_of": "2026-10-30",
    "coordination_level": "Urgent", "high_risk": False, "consult_status": "Active",
    "last_nurse_action_date": "2026-10-27"}})
threshold = (out.get("thresholds_used") or {}).get("no_action_days_urgent_complex")
smoke.append(("High-risk screen: Oncology urgent threshold is 2 days", (not err) and threshold == 2, s, threshold))

err, s, out = call_live("ask_vcare_case_data", {"args": {"question": "How many consults are flagged high-risk (High_Risk_Flag = Yes)?"}})
ok = (not err) and out.get("status") not in ("ERROR", "TIMEOUT") and "200" in json.dumps(out)
smoke.append(("Genie: 200 high-risk consults (also proves the app's data permissions)", ok, s,
              out.get("answer") or out.get("error") or out.get("status")))

for name, passed, seconds, detail in smoke:
    print(f"{'PASS' if passed else 'FAIL'}  {name}  ({seconds} s)  -> {str(detail)[:160]}")
SMOKE_PASSED = all(p for _, p, _, _ in smoke)
print("\nSmoke test:", "ALL PASS" if SMOKE_PASSED else "SOME CHECKS FAILED (see above); the full test below will still run")

In [0]:
# Same assistant loop as Option 1. The only difference: every tool call goes to the LIVE app.
# The AI model itself is called with YOUR normal notebook sign-in; only the app is reached as Inspector Gadget.
import xml.etree.ElementTree as ET
import mlflow
from mlflow.entities import SpanType

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(EXPERIMENT)
mlflow.openai.autolog()

gold = [
    {"q": i, "question": p.findtext("question"), "answer": p.findtext("answer"), "tool": EXPECTED_TOOL[i]}
    for i, p in enumerate(ET.parse(GOLD_SET).getroot().findall("qa_pair"), start=1)
]
assert len(gold) == 10, f"Expected 10 questions, found {len(gold)}"

def _input_schema(tool):
    """The tool's input format. Different MCP client versions name this attribute differently."""
    for attr in ("inputSchema", "input_schema", "parameters"):
        schema = getattr(tool, attr, None)
        if schema:
            return schema if isinstance(schema, dict) else json.loads(json.dumps(schema, default=lambda o: o.__dict__))
    dumped = tool.model_dump() if hasattr(tool, "model_dump") else vars(tool)
    for key in ("inputSchema", "input_schema", "parameters"):
        if dumped.get(key):
            return dumped[key]
    raise AttributeError(f"Could not find the input format for tool {tool.name}; available fields: {list(dumped)}")

TOOLS = [
    {"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": _input_schema(t)}}
    for t in live_tools
]
assert all("properties" in t["function"]["parameters"] for t in TOOLS), "A tool's input format looks empty"
llm = WorkspaceClient().serving_endpoints.get_open_ai_client()

def live_tool_text(name, args):
    """Run one tool on the live app and return its text. Errors are returned to the assistant, not raised."""
    try:
        result = call_tool_live(name, args)
        return result.content[0].text if result.content else ""
    except Exception as e:  # network errors arrive wrapped in an ExceptionGroup; report the real cause
        inner = getattr(e, "exceptions", [e])
        return json.dumps({"error": " | ".join(f"{type(x).__name__}: {x}" for x in inner)[:500]})

@mlflow.trace(span_type=SpanType.AGENT)
def answer_question(question: str) -> str:
    """Let the assistant work the question with the live tools, then return its final reply."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(8):   # at most 8 rounds of tool use per question
        msg = llm.chat.completions.create(model=AGENT_MODEL, messages=messages, tools=TOOLS, temperature=0).choices[0].message
        if not msg.tool_calls:
            return (msg.content or "").strip()
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            with mlflow.start_span(name=tc.function.name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                output = live_tool_text(tc.function.name, args)
                span.set_outputs(output[:4000])
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output[:12000]})
    return "(no final answer within 8 rounds)"

print(f"{len(gold)} questions loaded | tools given to the assistant: {[t['function']['name'] for t in TOOLS]}")

In [0]:
# 1) right tool used   2) exact final answer   3) judged correct by an independent AI grader
import re
from mlflow.entities import Feedback
from mlflow.genai.scorers import Correctness, scorer

def _normalize(text):
    """Compare letters and digits only (ignore case, spaces and punctuation)."""
    return re.sub(r"[^a-z0-9]", "", (text or "").lower())

@scorer
def used_expected_tool(trace, expectations) -> Feedback:
    used = [span.name for span in trace.search_spans(span_type=SpanType.TOOL)]
    ok = expectations["expected_tool"] in used
    return Feedback(value=ok, rationale=f"Tools called: {used or 'none'}. Needed: {expectations['expected_tool']}.")

@scorer
def final_answer_exact(outputs, expectations) -> Feedback:
    lines = [line for line in str(outputs).splitlines() if line.strip()]
    final_line = lines[-1] if lines else ""
    ok = _normalize(final_line) == _normalize(expectations["gold_answer"])
    return Feedback(value=ok, rationale=f"Final line {final_line!r} vs answer key {expectations['gold_answer']!r}.")

SCORERS = [used_expected_tool, final_answer_exact, Correctness()]

eval_rows = [
    {"inputs": {"question": g["question"]},
     "expectations": {"expected_facts": [g["answer"]], "gold_answer": g["answer"], "expected_tool": g["tool"]}}
    for g in gold
]
print(f"{len(eval_rows)} questions ready, {len(SCORERS)} checks each")

In [0]:
# Runs all 10 questions through the assistant against the LIVE app and grades them. About 3 to 5 minutes.
started = time.time()
with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params({"agent_model": AGENT_MODEL, "gold_set": "mcp_vcare_eval_0", "questions": len(eval_rows),
                       "transport": "live Databricks App over HTTPS, signed in as Inspector Gadget",
                       "app_url": APP_URL, "smoke_test_passed": SMOKE_PASSED})
    results = mlflow.genai.evaluate(data=eval_rows, predict_fn=answer_question, scorers=SCORERS)
    RUN_ID = run.info.run_id
ELAPSED = round(time.time() - started)
print(f"Finished in {ELAPSED} s | MLflow run {RUN_ID}")

In [0]:
# One row per question, then the totals, then a side-by-side with the practice run (Option 1).
import pandas as pd

traces = mlflow.search_traces(locations=[experiment.experiment_id], run_id=RUN_ID, return_type="list")

def _check(trace, name):
    for a in trace.info.assessments:
        if a.name == name and getattr(a, "feedback", None) is not None:
            return a.feedback.value in (True, "yes", "Yes", "true", 1)
    return False

by_question = {g["question"]: g for g in gold}
rows = []
for t in traces:
    q = json.loads(t.data.request)["question"]
    g = by_question[q]
    reply = t.data.response
    try:
        reply = json.loads(reply)
    except (TypeError, ValueError):
        pass
    rows.append({"q": g["q"], "tool needed": g["tool"], "answer key": g["answer"],
                 "assistant's final line": str(reply).strip().splitlines()[-1] if str(reply).strip() else "",
                 "right tool": _check(t, "used_expected_tool"), "exact answer": _check(t, "final_answer_exact"),
                 "judged correct": _check(t, "correctness"), "seconds": round(t.info.execution_duration / 1000, 1)})
scorecard = pd.DataFrame(rows).sort_values("q").reset_index(drop=True)
display(scorecard)

right_tool, exact, correct = (int(scorecard[c].sum()) for c in ("right tool", "exact answer", "judged correct"))
client = mlflow.MlflowClient()
for k, v in {"right_tool_count": right_tool, "exact_answer_count": exact, "judged_correct_count": correct,
             "elapsed_seconds": ELAPSED, "median_seconds_per_question": float(scorecard["seconds"].median())}.items():
    client.log_metric(RUN_ID, k, v)

print(f"LIVE APP    | right tool {right_tool}/10 | exact {exact}/10 | judged correct {correct}/10 | {ELAPSED} s total, median {scorecard['seconds'].median()} s per question")
print("PRACTICE    | right tool 10/10 | exact 10/10 | judged correct 10/10 | 104 s total, median 12.2 s per question (Option 1, Oct 30)")
goal = right_tool == 10 and correct == 10 and exact >= 9
print("\nGOAL MET: the live app matches the practice run." if goal else "\nGOAL NOT MET: open the MLflow traces for the failing questions.")
print(f"Saved to MLflow experiment {EXPERIMENT}, run {RUN_ID}")